In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

# ── 1. Load dataset ───────────────────────────────────────────────────────────
df = pd.read_csv("/ds_salaries.csv")
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print("\nFirst 3 rows:")
print(df.head(3).to_string())

# ── 2. Min-Max Normalisation of 'salary' ─────────────────────────────────────
print("\n" + "=" * 60)
print("MIN-MAX NORMALISATION — salary column")
print("=" * 60)

scaler = MinMaxScaler()
df["salary_normalized"] = scaler.fit_transform(df[["salary"]])

print(f"Original  → min: {df['salary'].min():,.0f}  max: {df['salary'].max():,.0f}")
print(f"Normalised→ min: {df['salary_normalized'].min():.4f}  max: {df['salary_normalized'].max():.4f}")
print("\nSample comparison:")
print(df[["salary", "salary_normalized"]].head(5).to_string())

# ── 3. PCA dimensionality reduction ──────────────────────────────────────────
print("\n" + "=" * 60)
print("PCA — DIMENSIONALITY REDUCTION")
print("=" * 60)

# Encode all categorical columns so PCA can work on a numeric matrix
df_enc = df.copy()
le = LabelEncoder()
cat_cols = df_enc.select_dtypes(include="object").columns
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

# Use the numeric features (drop the target salary columns to keep it clean)
feature_cols = [c for c in df_enc.columns if c not in ("salary", "salary_normalized", "salary_in_usd")]
X = df_enc[feature_cols].fillna(0)

# Standardise before PCA (PCA is scale-sensitive)
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)

df["PC1"] = pcs[:, 0]
df["PC2"] = pcs[:, 1]

print(f"Original feature count : {X.shape[1]}")
print(f"After PCA              : 2 components")
print(f"Variance explained     : PC1={pca.explained_variance_ratio_[0]:.1%}  "
      f"PC2={pca.explained_variance_ratio_[1]:.1%}  "
      f"Total={sum(pca.explained_variance_ratio_):.1%}")
print("\nPCA loadings (top features per component):")
loadings = pd.DataFrame(
    pca.components_.T, index=feature_cols, columns=["PC1", "PC2"]
)
print("  PC1 top 3:", loadings["PC1"].abs().nlargest(3).index.tolist())
print("  PC2 top 3:", loadings["PC2"].abs().nlargest(3).index.tolist())

# ── 4. Group by experience_level → avg & median salary ───────────────────────
print("\n" + "=" * 60)
print("SALARY AGGREGATION BY EXPERIENCE LEVEL")
print("=" * 60)

exp_map = {"EN": "Entry-level", "MI": "Mid-level", "SE": "Senior", "EX": "Executive/Expert"}
df["experience_label"] = df["experience_level"].map(exp_map).fillna(df["experience_level"])

agg = (
    df.groupby("experience_label")["salary"]
    .agg(Count="count", Average="mean", Median="median")
    .sort_values("Average")
)
agg["Average"] = agg["Average"].map("${:,.0f}".format)
agg["Median"]  = agg["Median"].map("${:,.0f}".format)
print(agg.to_string())

# ── 5. Save enriched dataset ──────────────────────────────────────────────────
out = "/mnt/user-data/outputs/salary_analysis_results.csv"
df.to_csv(out, index=False)
print("\n" + "=" * 60)
print(f" Enriched dataset saved → {out}")
print("   New columns added: salary_normalized, PC1, PC2, experience_label")
print("=" * 60)